# Sentiment classification

Sentiment classification assigns a single label — positive or negative — to a whole document. A recurrent model reads the sequence word by word and uses the final hidden state as a summary. This notebook builds the pipeline on the NLTK movie reviews corpus: a logistic baseline over bag-of-words, then an LSTM classifier in PyTorch, then the same LSTM initialised with pre-trained word embeddings.

Order: intuition, maths, data, baseline, LSTM, pre-trained embeddings, tuning, what breaks.

In [ ]:
import random
from collections import Counter

import nltk
import numpy as np
import torch
from torch import nn
from torch.utils.data import DataLoader, Dataset

nltk.download('movie_reviews', quiet=True)
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = torch.device('mps' if torch.backends.mps.is_available() else 'cpu')
print(f'device: {DEVICE}')

## 1. Intuition

Classification of a whole document is harder than predicting the next token. The model receives hundreds of words but produces only one bit of signal — positive or negative — so gradients are diluted across the entire sequence. A bag-of-words model ignores order entirely and counts words. It is surprisingly strong on sentiment because adjectives carry most of the information. An LSTM can capture order and negation, but needs enough data to outperform the simpler baseline.

## 2. Maths

The LSTM recurrence is unchanged from notebook 04: $\mathbf{h}_t, \mathbf{c}_t = \text{LSTM}(\mathbf{x}_t, \mathbf{h}_{t-1}, \mathbf{c}_{t-1})$. For classification we keep only the last valid hidden state $\mathbf{h}_T$ and pass it through a linear layer: $\hat{y} = \sigma(\mathbf{W}\mathbf{h}_T + b)$. The loss is binary cross-entropy:

$$\mathcal{L} = -\frac{1}{n}\sum_{i=1}^n \bigl[ y_i \log \hat{y}_i + (1 - y_i) \log(1 - \hat{y}_i) \bigr]$$

Accuracy and F1 are both reported; the dataset is balanced so accuracy is not misleading here.

## 3. Data

In [ ]:
from nltk.corpus import movie_reviews
from nltk.tokenize import word_tokenize

MAX_LEN = 300
VOCAB_SIZE = 5000

# Load all reviews as token lists with labels
docs = []
for label, category in [(1, 'pos'), (0, 'neg')]:
    for fileid in movie_reviews.fileids(category):
        raw = movie_reviews.raw(fileid).lower()
        tokens = word_tokenize(raw)
        docs.append((tokens, label))

rng = np.random.default_rng(SEED)
idx = rng.permutation(len(docs))
docs = [docs[i] for i in idx]

print(f'{len(docs)} documents, {sum(y for _, y in docs)} positive')

In [ ]:
# Build vocab from training split only to avoid leakage
split = int(0.8 * len(docs))
train_docs, test_docs = docs[:split], docs[split:]

counter = Counter(tok for tokens, _ in train_docs for tok in tokens)
# index 0 = <PAD>, index 1 = <OOV>, words start at 2
vocab = {'<PAD>': 0, '<OOV>': 1}
for word, _ in counter.most_common(VOCAB_SIZE - 2):
    vocab[word] = len(vocab)

def encode(tokens: list[str], max_len: int) -> list[int]:
    ids = [vocab.get(t, 1) for t in tokens[:max_len]]
    ids += [0] * (max_len - len(ids))  # pad to max_len
    return ids

print(f'vocab size: {len(vocab)}')
print(f'train: {len(train_docs)}, test: {len(test_docs)}')

In [ ]:
class ReviewDataset(Dataset):
    def __init__(self, docs: list[tuple[list[str], int]], max_len: int) -> None:
        self.sequences = [encode(tokens, max_len) for tokens, _ in docs]
        self.lengths = [min(len(tokens), max_len) for tokens, _ in docs]
        self.labels = [label for _, label in docs]

    def __len__(self) -> int:
        return len(self.labels)

    def __getitem__(self, i: int) -> tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
        return (
            torch.tensor(self.sequences[i], dtype=torch.long),
            torch.tensor(self.labels[i], dtype=torch.float),
            torch.tensor(self.lengths[i], dtype=torch.long),
        )

train_ds = ReviewDataset(train_docs, MAX_LEN)
test_ds = ReviewDataset(test_docs, MAX_LEN)
train_loader = DataLoader(train_ds, batch_size=64, shuffle=True)
test_loader = DataLoader(test_ds, batch_size=64)

## 4. Logistic baseline

Bag-of-words with logistic regression: each review becomes a TF-IDF vector and the model classifies it with no knowledge of word order. This is the bar the LSTM needs to beat.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score

train_texts = [' '.join(tokens) for tokens, _ in train_docs]
test_texts  = [' '.join(tokens) for tokens, _ in test_docs]
train_labels = [y for _, y in train_docs]
test_labels  = [y for _, y in test_docs]

tfidf = TfidfVectorizer(max_features=VOCAB_SIZE)
X_train = tfidf.fit_transform(train_texts)
X_test  = tfidf.transform(test_texts)

lr = LogisticRegression(max_iter=1000, random_state=SEED)
lr.fit(X_train, train_labels)
preds = lr.predict(X_test)

print(f'Logistic — accuracy: {accuracy_score(test_labels, preds):.3f}, F1: {f1_score(test_labels, preds):.3f}')

## 5. LSTM classifier

In [ ]:
class SentimentLSTM(nn.Module):
    def __init__(self, vocab_size: int, embedding_dim: int, hidden_size: int, pad_idx: int = 0) -> None:
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=pad_idx)
        self.lstm = nn.LSTM(embedding_dim, hidden_size, batch_first=True)
        self.dropout = nn.Dropout(0.3)
        self.fc = nn.Linear(hidden_size, 1)

    def forward(self, x: torch.Tensor, lengths: torch.Tensor) -> torch.Tensor:
        emb = self.dropout(self.embedding(x))  # (batch, seq, emb)
        packed = nn.utils.rnn.pack_padded_sequence(
            emb, lengths.cpu(), batch_first=True, enforce_sorted=False
        )
        _, (h_n, _) = self.lstm(packed)  # h_n: (1, batch, hidden)
        h_last = self.dropout(h_n.squeeze(0))  # (batch, hidden)
        return self.fc(h_last).squeeze(1)  # (batch,)

In [ ]:
def train_model(
    model: nn.Module,
    loader: DataLoader,
    epochs: int = 10,
    lr: float = 1e-3,
) -> list[float]:
    model.to(DEVICE)
    optimiser = torch.optim.Adam(model.parameters(), lr=lr)
    criterion = nn.BCEWithLogitsLoss()
    losses = []
    for epoch in range(epochs):
        model.train()
        total = 0.0
        for x, y, lengths in loader:
            x, y = x.to(DEVICE), y.to(DEVICE)
            optimiser.zero_grad()
            logits = model(x, lengths)
            loss = criterion(logits, y)
            loss.backward()
            optimiser.step()
            total += loss.item()
        losses.append(total / len(loader))
        print(f'epoch {epoch+1:2d}  loss={losses[-1]:.4f}')
    return losses


def evaluate(model: nn.Module, loader: DataLoader) -> tuple[float, float]:
    model.eval()
    all_preds, all_labels = [], []
    with torch.no_grad():
        for x, y, lengths in loader:
            x = x.to(DEVICE)
            logits = model(x, lengths)
            preds = (torch.sigmoid(logits) >= 0.5).long().cpu().tolist()
            all_preds.extend(preds)
            all_labels.extend(y.long().tolist())
    return accuracy_score(all_labels, all_preds), f1_score(all_labels, all_preds)

In [ ]:
EMBEDDING_DIM = 50
HIDDEN_SIZE = 64

lstm_model = SentimentLSTM(len(vocab), EMBEDDING_DIM, HIDDEN_SIZE)
train_model(lstm_model, train_loader, epochs=10)
acc, f1 = evaluate(lstm_model, test_loader)
print(f'LSTM (random init) — accuracy: {acc:.3f}, F1: {f1:.3f}')

## 6. Pre-trained embeddings

Instead of starting from random, we train a small Word2Vec model on the movie reviews corpus and load those vectors into the embedding layer. The embeddings are frozen for the first 5 epochs so the LSTM adapts without immediately overwriting the pre-trained geometry, then unfrozen to allow fine-tuning.

In [ ]:
from gensim.models import Word2Vec

all_tokens = [tokens for tokens, _ in docs]
w2v = Word2Vec(sentences=all_tokens, vector_size=EMBEDDING_DIM, window=3, min_count=2,
               workers=1, seed=SEED, epochs=5)

# Build a weight matrix aligned to our vocab
pretrained = np.zeros((len(vocab), EMBEDDING_DIM), dtype=np.float32)
hits = 0
for word, idx in vocab.items():
    if word in w2v.wv:
        pretrained[idx] = w2v.wv[word]
        hits += 1
print(f'Loaded {hits}/{len(vocab)} word vectors from Word2Vec')

In [ ]:
lstm_pretrained = SentimentLSTM(len(vocab), EMBEDDING_DIM, HIDDEN_SIZE)
lstm_pretrained.embedding.weight = nn.Parameter(torch.tensor(pretrained))
lstm_pretrained.embedding.weight.requires_grad = False  # freeze

# Train with frozen embeddings for 5 epochs
print('--- frozen embeddings ---')
train_model(lstm_pretrained, train_loader, epochs=5)

# Unfreeze and train 5 more
lstm_pretrained.embedding.weight.requires_grad = True
print('--- fine-tuning embeddings ---')
train_model(lstm_pretrained, train_loader, epochs=5)

acc_pt, f1_pt = evaluate(lstm_pretrained, test_loader)
print(f'LSTM (pre-trained) — accuracy: {acc_pt:.3f}, F1: {f1_pt:.3f}')
print(f'Random init F1: {f1:.3f} → Pre-trained F1: {f1_pt:.3f}')

## 7. Tuning

- **Dropout**: 0.3 on both the embedding output and the final hidden state reduces overfitting on small corpora. Values above 0.5 hurt convergence.
- **GRU vs LSTM**: a GRU has fewer parameters and trains about 20% faster here with similar accuracy. LSTM is worth the cost only when long-range dependencies matter — 300-token reviews sit near the edge.
- **Frozen vs fine-tuned embeddings**: freezing for the first few epochs prevents the LSTM from collapsing the pre-trained geometry before it has learned anything. Fine-tuning after stabilisation usually adds 1–2 F1 points.

## 8. What breaks

- **Negation**: "not good" and "good" produce nearly identical bag-of-words representations. An LSTM can in principle learn to handle `not`, but only if negation appears frequently and consistently in training.
- **Sarcasm**: the surface words are positive while the meaning is negative. No model here captures that — it requires world knowledge or pragmatic context.
- **Domain shift**: a model trained on movie reviews performs poorly on product reviews or tweets. Sentiment words are domain-specific ("unpredictable plot" is positive in films, meaningless elsewhere).
- **Class imbalance**: movie_reviews is balanced, but real-world sentiment data rarely is. Accuracy becomes a misleading metric when 95% of reviews are neutral.